# PCA & Manifold Learning — Concepts
**Dimensionality Reduction: Data ko Compress karo bina Information Khoye**

---
## Table of Contents
1. PCA kya hai? — Intuition & Why we need it
2. Variance, Covariance aur Eigenvalues — Math behind PCA
3. Manual PCA — Step by Step (Small Dataset)
4. Sklearn PCA — Real Implementation
5. Explained Variance Ratio — Kitne Components rakhein?
6. t-SNE — Manifold Learning ka Introduction
7. Capstone Challenge + Solutions
---

## Section 1: PCA kya hai?

### Real Life Example:
Socho tumhare paas ek student ka record hai:
- Math marks, Physics marks, Chemistry marks (ye teeno highly correlated hain)
- Height, Weight
- Attendance, Homework completion (ye dono correlated hain)

**Problem:** Bahut saari features hain. Kuch features ek saath move karti hain (correlated). Iska matlab hai hum redundant information le rahe hain.

**PCA ka Solution:** 
- Correlated features ko combine karke **Principal Components** banao
- Pehla component sabse zyada variance capture karta hai
- Dusra component pehle se perpendicular hota hai, uske baad sabse zyada variance
- Hum sirf top-k components rakhte hain

**Fayda:**
- 100 features → 2-3 features (visualization possible!)
- Noise reduce hoti hai
- Training fast hoti hai
- Overfitting kam hoti hai

**Keywords:**
- **Principal Component (PC):** Naya axis jo maximum variance direction mein hai
- **Dimensionality Reduction:** Features ka number kam karna
- **Variance:** Data points ka spread (kitne door-door hain)
- **Orthogonal:** 90 degree angle — PCs ek doosre se independent hote hain

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# PCA ka Visual Intuition
# 2D data banate hain jisme features correlated hain
np.random.seed(42)
n = 100
x1 = np.random.randn(n)
x2 = 2 * x1 + 0.5 * np.random.randn(n)  # x2 is correlated with x1

plt.figure(figsize=(12, 4))

# Original correlated data
plt.subplot(1, 2, 1)
plt.scatter(x1, x2, alpha=0.6, color='steelblue')
plt.xlabel('Feature 1 (x1)')
plt.ylabel('Feature 2 (x2)')
plt.title('Original Data\n(Highly Correlated!)')

# PCA direction show karo
mean = np.array([x1.mean(), x2.mean()])
cov = np.cov(np.vstack([x1, x2]))
eigenvalues, eigenvectors = np.linalg.eig(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

scale = 2
plt.subplot(1, 2, 2)
plt.scatter(x1, x2, alpha=0.3, color='steelblue')
colors = ['red', 'green']
labels = ['PC1 (max variance)', 'PC2 (perpendicular)']
for i in range(2):
    v = eigenvectors[:, i] * scale * np.sqrt(eigenvalues[i])
    plt.arrow(mean[0], mean[1], v[0], v[1], head_width=0.1,
              head_length=0.1, fc=colors[i], ec=colors[i], linewidth=2, label=labels[i])
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('PCA Principal Components')
plt.legend()
plt.tight_layout()
plt.show()

print(f"PC1 direction: {eigenvectors[:, 0].round(3)}")
print(f"PC2 direction: {eigenvectors[:, 1].round(3)}")
print(f"PC1 captures {eigenvalues[0]/eigenvalues.sum()*100:.1f}% variance")
print(f"PC2 captures {eigenvalues[1]/eigenvalues.sum()*100:.1f}% variance")

## Section 2: Variance, Covariance aur Eigenvalues

### Keywords:
- **Variance (σ²):** Ek feature ka spread — `var(x) = Σ(xᵢ - mean)² / n`
- **Covariance:** Do features ke beech relationship — `cov(x,y) = Σ(xᵢ-x̄)(yᵢ-ȳ) / n`
  - Positive: dono saath badhte hain
  - Negative: ek badhta hai, doosra ghatta hai
  - Zero: independent
- **Covariance Matrix:** Saari features ki covariance ek matrix mein
- **Eigenvalue (λ):** Batata hai PC kitna variance capture karta hai
- **Eigenvector:** Batata hai PC ka direction kya hai
- **Standardization:** PCA se pehle features ko same scale pe lana zaroori hai!

In [ ]:
import numpy as np

# Small 4-student dataset: Math, Physics, Chemistry scores
data = np.array([
    [85, 80, 83],   # Student A
    [60, 55, 58],   # Student B
    [90, 88, 91],   # Student C
    [70, 65, 68],   # Student D
])

print("=== Step 1: Original Data ===")
print("Student | Math | Physics | Chemistry")
names = ['A', 'B', 'C', 'D']
for i, row in enumerate(data):
    print(f"   {names[i]}    |  {row[0]}  |   {row[1]}   |    {row[2]}")

# Step 2: Standardize (mean=0, std=1)
mean = data.mean(axis=0)
std = data.std(axis=0)
data_std = (data - mean) / std
print(f"\n=== Step 2: Standardize ===")
print(f"Mean: {mean.round(1)}")
print(f"Std:  {std.round(1)}")
print(f"Standardized data:\n{data_std.round(3)}")

# Step 3: Covariance Matrix
cov_matrix = np.cov(data_std.T)
print(f"\n=== Step 3: Covariance Matrix ===")
print(cov_matrix.round(3))
print("(Diagonal = variance of each feature, Off-diagonal = covariance between features)")

# Step 4: Eigenvalues & Eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
# Sort by eigenvalue descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx].real
eigenvectors = eigenvectors[:, idx].real

print(f"\n=== Step 4: Eigenvalues (how much variance each PC captures) ===")
for i, ev in enumerate(eigenvalues):
    pct = ev / eigenvalues.sum() * 100
    print(f"PC{i+1}: eigenvalue = {ev:.3f}  → captures {pct:.1f}% variance")

print(f"\n=== Step 5: Eigenvectors (direction of each PC) ===")
features = ['Math', 'Physics', 'Chemistry']
for i in range(len(eigenvalues)):
    print(f"PC{i+1}: {dict(zip(features, eigenvectors[:, i].round(3)))}")

# Step 6: Transform data
data_pca = data_std @ eigenvectors
print(f"\n=== Step 6: Transformed Data (3D → 3D, PC space) ===")
print("Student |  PC1   |  PC2   |  PC3")
for i, row in enumerate(data_pca):
    print(f"   {names[i]}    | {row[0]:+.3f} | {row[1]:+.3f} | {row[2]:+.3f}")

print("\n→ If we keep only PC1, we capture", f"{eigenvalues[0]/eigenvalues.sum()*100:.1f}% of information!")

## Section 3: Manual PCA Visualization

In [ ]:
import matplotlib.pyplot as plt

# Visualize the transformation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original features
axes[0].scatter(data_std[:, 0], data_std[:, 1], s=150, c='steelblue', zorder=5)
for i, name in enumerate(names):
    axes[0].annotate(f'  Student {name}', (data_std[i, 0], data_std[i, 1]), fontsize=11)
axes[0].set_xlabel('Math (standardized)')
axes[0].set_ylabel('Physics (standardized)')
axes[0].set_title('Original Feature Space\n(Math vs Physics — Correlated!)')
axes[0].axhline(0, color='gray', lw=0.5)
axes[0].axvline(0, color='gray', lw=0.5)

# PCA space
axes[1].scatter(data_pca[:, 0], data_pca[:, 1], s=150, c='coral', zorder=5)
for i, name in enumerate(names):
    axes[1].annotate(f'  Student {name}', (data_pca[i, 0], data_pca[i, 1]), fontsize=11)
axes[1].set_xlabel('PC1 (most variance)')
axes[1].set_ylabel('PC2 (remaining variance)')
axes[1].set_title('PCA Space\n(Independent Components!)')
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].axvline(0, color='gray', lw=0.5)

plt.tight_layout()
plt.show()

print("\nAaj humne kya seekha:")
print("- Math, Physics, Chemistry — teeno correlated the")
print("- PCA ne inhe PC1 (overall performance) aur PC2 (Math vs Science lean) mein convert kiya")
print("- PC1 ek axis hai jisme students sabse zyada spread hain")
print("- Agar hum sirf PC1 rakhein, toh 3D → 1D ho jaata hai with minimal info loss!")

## Section 4: Sklearn PCA — Real Implementation

### Keywords:
- **`PCA(n_components=k)`:** k principal components banao
- **`.fit_transform(X)`:** Fit karo aur transform karo ek saath
- **`.explained_variance_ratio_`:** Har PC kitna variance capture karta hai (0 to 1)
- **`n_components=0.95`:** 95% variance explain karne ke liye enough components automatically
- **Scree Plot:** Eigenvalues ka graph — kahan 'elbow' hai wahan cut karo

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris
import numpy as np
import matplotlib.pyplot as plt

# Iris dataset — 4 features, 3 classes
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
class_names = iris.target_names

print(f"Original shape: {X.shape}")
print(f"Features: {feature_names}")

# Step 1: Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: PCA with all components first
pca_full = PCA()
pca_full.fit(X_scaled)

print("\n=== Explained Variance Ratio ===")
for i, var in enumerate(pca_full.explained_variance_ratio_):
    cumsum = pca_full.explained_variance_ratio_[:i+1].sum()
    print(f"PC{i+1}: {var*100:.1f}% variance  (Cumulative: {cumsum*100:.1f}%)")

# Scree Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 5), pca_full.explained_variance_ratio_ * 100, color='steelblue', alpha=0.7)
axes[0].plot(range(1, 5), np.cumsum(pca_full.explained_variance_ratio_ * 100), 
             'ro-', linewidth=2, markersize=8, label='Cumulative')
axes[0].axhline(95, color='green', linestyle='--', label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot — Kitne PC rakhein?')
axes[0].legend()

# PCA 2D visualization
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_scaled)

colors = ['red', 'green', 'blue']
for i, (cls, color) in enumerate(zip(class_names, colors)):
    mask = y == i
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=cls, alpha=0.7, s=60)

axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)')
axes[1].set_title('Iris Dataset: 4D → 2D via PCA')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n4D → 2D PCA keeps {pca_2d.explained_variance_ratio_.sum()*100:.1f}% of information!")

In [ ]:
# Feature Loadings — PC mein kaunsi feature ka contribution zyada hai?
loadings = pca_2d.components_

print("=== Feature Loadings (Contribution to each PC) ===")
print(f"{'Feature':<25} {'PC1':>8} {'PC2':>8}")
print('-' * 45)
for feat, l1, l2 in zip(feature_names, loadings[0], loadings[1]):
    print(f"{feat:<25} {l1:>+8.3f} {l2:>+8.3f}")

print("\nInterpretation:")
print("- PC1 mein sab features ka similar loading hai → 'Overall Size'")
print("- PC2 mein sepal vs petal alag direction hai → 'Shape Contrast'")

# PCA with 95% variance
pca_95 = PCA(n_components=0.95)
X_95 = pca_95.fit_transform(X_scaled)
print(f"\nPCA with 95% variance threshold:")
print(f"  Number of components needed: {pca_95.n_components_}")
print(f"  Total variance captured: {pca_95.explained_variance_ratio_.sum()*100:.1f}%")

## Section 5: PCA with Classification — Before & After Comparison

### Kab PCA use karein?
- Bahut saari features hain (100+)
- Features mein multicollinearity hai
- 2D/3D visualization chahiye
- Training slow hai

### Caution!
- PCA se interpretability kho jaati hai
- Always standardize before PCA
- Test data pe **sirf transform** karo, fit nahi

In [ ]:
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Wine dataset — 13 features
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

print(f"Wine dataset: {X_wine.shape[0]} samples, {X_wine.shape[1]} features")

# Pipeline: Scale → (optional PCA) → Classify
results = {}

# Without PCA
pipe_no_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])
scores_no_pca = cross_val_score(pipe_no_pca, X_wine, y_wine, cv=5)
results['No PCA (13 features)'] = scores_no_pca.mean()

# With different PCA components
for n in [2, 5, 8, 10]:
    pipe_pca = Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n)),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    scores = cross_val_score(pipe_pca, X_wine, y_wine, cv=5)
    results[f'PCA ({n} components)'] = scores.mean()

print("\n=== Accuracy Comparison ===")
for name, acc in results.items():
    bar = '█' * int(acc * 40)
    print(f"{name:<25} {acc:.3f} {bar}")

print("\n→ PCA with 5-8 components gives almost same accuracy with 50-60% fewer features!")

## Section 6: t-SNE — Manifold Learning

### t-SNE kya hai?
PCA linear hai — sirf straight lines mein project karta hai. 
Real world data non-linear hota hai — **t-SNE (t-distributed Stochastic Neighbor Embedding)** non-linear structure ko preserve karta hai.

### Keywords:
- **Manifold:** High-dimensional data jo actually low-dimensional curved surface pe hai
- **t-SNE:** High-D mein similar points ko Low-D mein bhi paas rakhta hai
- **Perplexity:** Approximately kitne neighbors consider karein (5-50, default 30)
- **Limitation:** t-SNE sirf visualization ke liye hai, prediction ke liye nahi
- **Stochastic:** Har run mein alag result aa sakta hai

### PCA vs t-SNE:
| Feature | PCA | t-SNE |
|---------|-----|-------|
| Type | Linear | Non-linear |
| Speed | Fast | Slow |
| Use | Preprocessing + ML | Visualization only |
| Reproducible | Yes | Need random_state |
| Interpretable | Yes (loadings) | No |

In [ ]:
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt
import numpy as np

# Digits dataset — 1797 images, 64 features (8x8 pixels)
digits = load_digits()
X_dig = digits.data   # 64 features
y_dig = digits.target  # 0-9

print(f"Digits dataset: {X_dig.shape} → 64 pixel features, 10 classes (0-9)")

# Use subset for speed
X_sub = X_dig[:500]
y_sub = y_dig[:500]

# PCA 2D
pca_dig = PCA(n_components=2)
X_pca_dig = pca_dig.fit_transform(StandardScaler().fit_transform(X_sub))

# t-SNE 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cmap = plt.cm.get_cmap('tab10', 10)
for digit in range(10):
    mask = y_sub == digit
    axes[0].scatter(X_pca_dig[mask, 0], X_pca_dig[mask, 1], 
                    c=[cmap(digit)], label=str(digit), alpha=0.7, s=30)
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                    c=[cmap(digit)], label=str(digit), alpha=0.7, s=30)

axes[0].set_title('PCA: 64D → 2D\n(Linear — Clusters overlap)')
axes[0].legend(title='Digit', bbox_to_anchor=(1, 1), fontsize=8)

axes[1].set_title('t-SNE: 64D → 2D\n(Non-linear — Better clusters!)')
axes[1].legend(title='Digit', bbox_to_anchor=(1, 1), fontsize=8)

plt.tight_layout()
plt.show()

print("Observation:")
print("- PCA: Clusters overlap — kuch digits mix ho jaate hain")
print("- t-SNE: Har digit ka alag cluster banta hai — much clearer!")
print("- t-SNE is better for visualization, PCA for ML preprocessing")

## Section 7: Aaj Humne Kya Seekha?

### Summary:
1. **PCA** — Correlated features ko independent Principal Components mein convert karta hai
2. **Steps:** Standardize → Covariance Matrix → Eigenvalues/Eigenvectors → Project
3. **Scree Plot** — Elbow method se decide karo kitne PCs rakhne hain
4. **Explained Variance Ratio** — Har PC kitna information rakhta hai
5. **t-SNE** — Non-linear visualization ke liye (ML ke liye nahi!)

### Real-World Uses:
- **Face Recognition (Eigenfaces):** 100x100 image = 10,000 pixels → PCA se 100 components
- **Genomics:** Thousands of genes → PCA se population structure visualize
- **Finance:** 500 stocks → PCA se market factors (Growth, Value, Momentum)
- **NLP:** Word embeddings visualization with t-SNE
- **Medical Imaging:** MRI/CT scan compression

---
## Capstone Challenge

**Problem:** Breast Cancer Wisconsin dataset pe PCA lagao.

**Tasks:**
1. Load `load_breast_cancer()` from sklearn
2. Standardize the data
3. Apply PCA with all components and plot the **scree plot**
4. How many components are needed to capture **90% variance**?
5. Apply PCA with 2 components and visualize malignant vs benign
6. Compare Logistic Regression accuracy: Original features vs PCA features
7. Apply t-SNE with 2 components and compare with PCA visualization

**Dataset:** Breast Cancer Wisconsin — 30 features, 569 samples, 2 classes (malignant/benign)

In [ ]:
# CAPSTONE — TRY IT YOURSELF FIRST!
# Scroll down for solutions

from sklearn.datasets import load_breast_cancer

bc = load_breast_cancer()
X_bc = bc.data
y_bc = bc.target

print(f"Dataset: {X_bc.shape}, Classes: {bc.target_names}")
print("Your turn! Try the 7 tasks above...")

# YOUR CODE HERE


---
## Solutions — Capstone

In [ ]:
# === SOLUTION ===
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
print(f"Dataset: {X_bc.shape} — {bc.target_names}")

# Task 2: Standardize
scaler = StandardScaler()
X_bc_scaled = scaler.fit_transform(X_bc)

# Task 3: PCA with all components
pca_all = PCA()
pca_all.fit(X_bc_scaled)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Scree plot
cumvar = np.cumsum(pca_all.explained_variance_ratio_) * 100
axes[0, 0].bar(range(1, 31), pca_all.explained_variance_ratio_ * 100, alpha=0.7, color='steelblue')
axes[0, 0].plot(range(1, 31), cumvar, 'ro-', markersize=4, label='Cumulative')
axes[0, 0].axhline(90, color='green', linestyle='--', label='90% threshold')
axes[0, 0].set_xlabel('Principal Component')
axes[0, 0].set_ylabel('Explained Variance (%)')
axes[0, 0].set_title('Scree Plot')
axes[0, 0].legend()

# Task 4: Components for 90% variance
n_90 = np.argmax(cumvar >= 90) + 1
print(f"\nTask 4: {n_90} components needed for 90% variance")

# Task 5: PCA 2D visualization
pca_2d = PCA(n_components=2)
X_bc_2d = pca_2d.fit_transform(X_bc_scaled)

colors = ['salmon', 'steelblue']
for i, (cls, color) in enumerate(zip(bc.target_names, colors)):
    mask = y_bc == i
    axes[0, 1].scatter(X_bc_2d[mask, 0], X_bc_2d[mask, 1], c=color, label=cls, alpha=0.6, s=40)
axes[0, 1].set_title(f'PCA 2D\n({pca_2d.explained_variance_ratio_.sum()*100:.1f}% variance)')
axes[0, 1].set_xlabel('PC1')
axes[0, 1].set_ylabel('PC2')
axes[0, 1].legend()

# Task 6: Accuracy comparison
pipe_orig = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
acc_orig = cross_val_score(pipe_orig, X_bc, y_bc, cv=5).mean()

pipe_pca = Pipeline([
    ('sc', StandardScaler()),
    ('pca', PCA(n_components=n_90)),
    ('clf', LogisticRegression(max_iter=1000))
])
acc_pca = cross_val_score(pipe_pca, X_bc, y_bc, cv=5).mean()

models = ['Original\n30 features', f'PCA\n{n_90} features']
accs = [acc_orig, acc_pca]
axes[1, 0].bar(models, accs, color=['steelblue', 'coral'], alpha=0.8)
for i, acc in enumerate(accs):
    axes[1, 0].text(i, acc + 0.002, f'{acc:.3f}', ha='center', fontweight='bold')
axes[1, 0].set_ylim(0.9, 1.01)
axes[1, 0].set_title('Accuracy: Original vs PCA')
axes[1, 0].set_ylabel('Cross-Val Accuracy')
print(f"Task 6: Original accuracy = {acc_orig:.3f}, PCA accuracy = {acc_pca:.3f}")

# Task 7: t-SNE
tsne_bc = TSNE(n_components=2, random_state=42, perplexity=30)
X_bc_tsne = tsne_bc.fit_transform(X_bc_scaled)

for i, (cls, color) in enumerate(zip(bc.target_names, colors)):
    mask = y_bc == i
    axes[1, 1].scatter(X_bc_tsne[mask, 0], X_bc_tsne[mask, 1], c=color, label=cls, alpha=0.6, s=40)
axes[1, 1].set_title('t-SNE 2D\n(Better separation!)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"- 30 features → {n_90} components needed for 90% variance")
print(f"- Accuracy barely drops: {acc_orig:.3f} → {acc_pca:.3f}")
print(f"- t-SNE shows cleaner separation than PCA for visualization")